<a href="https://colab.research.google.com/github/MakotoMurakami2004/Aplicativo-TCC/blob/main/TreinamentoMachineLearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas openpyxl "numpy<2.0.0" "scikit-learn==1.3.2" joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 27.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 60.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> scikit-learn

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [2]:
import os
import re
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from google.colab import files

In [3]:
# 2. Leitura das abas da planilha Excel
df_veridicos = pd.read_excel("NovaTabela de Links.xlsx", sheet_name=0)
df_suspeitos = pd.read_excel("NovaTabela de Links.xlsx", sheet_name=1)

# Limpeza e padronização da coluna de URLs
df_veridicos['url'] = df_veridicos['url'].astype(str).str.strip()
df_suspeitos['url'] = df_suspeitos['url'].astype(str).str.strip()

In [4]:
# 3. Função de Extração de Características (28 métricas - Idêntica ao Android Studio)
def extract_features(url):
    """Analisa o texto da URL e extrai métricas numéricas para a IA."""
    url = str(url).strip().lower()

    if not url.startswith(("http://", "https://")):
        url = "https://" + url

    return {
        "tamanho_url": len(url),
        "qtd_pontos": url.count("."),
        "qtd_hifens": url.count("-"),
        "qtd_arroba": url.count("@"),
        "qtd_interrogacao": url.count("?"),
        "qtd_igual": url.count("="),
        # Conta barras ignorando o protocolo e a barra final
        "qtd_barras": url.replace("https://", "").replace("http://", "").strip("/").count("/"),
        "qtd_digitos": sum(c.isdigit() for c in url),
        "tem_https": int(url.startswith("https://")),
        "tem_ip": int(re.search(r"\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b", url) is not None),
        "qtd_subdominios": url.count("."),

        # Indicadores institucionais / confiáveis
        "tem_gov": int("gov" in url),
        "tem_edu": int("edu" in url),
        "tem_org": int("org" in url),
        "tem_br": int(".br" in url),
        "tem_fgv": int("fgv" in url),
        "tem_ufrn": int("ufrn" in url),
        "tem_unesp": int("unesp" in url),
        "tem_ufsc": int("ufsc" in url),
        "tem_ufrj": int("ufrj" in url),

        # Palavras comuns em phishing
        "tem_login": int("login" in url),
        "tem_verify": int("verify" in url),
        "tem_secure": int("secure" in url),
        "tem_account": int("account" in url),
        "tem_update": int("update" in url),
        "tem_bank": int("bank" in url),
        "tem_paypal": int("paypal" in url),
        "tem_free": int("free" in url),
    }

In [5]:
# 4. Rotulação e Preparação dos Dados
df_veridicos["label"] = 0  # 0 = Legítimo
df_suspeitos["label"] = 1  # 1 = Phishing

df_urls = pd.concat([df_veridicos, df_suspeitos], ignore_index=True)

X = pd.DataFrame(df_urls["url"].apply(extract_features).tolist())
y = df_urls["label"]

# 5. Divisão Treino/Teste (80% Treino, 20% Teste com estratificação)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 6. Treinamento do Modelo nos Dados de Treino
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [7]:
# 7. Avaliação nos Dados de Teste
y_pred = modelo.predict(X_test)

acuracia = accuracy_score(y_test, y_pred)
precisao = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
matriz = confusion_matrix(y_test, y_pred)

print("=" * 50)
print("      MÉTRICAS DE DESEMPENHO")
print("=" * 50)
print(f"Acurácia:  {acuracia * 100:.2f}%")
print(f"Precisão:  {precisao * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-Score:  {f1 * 100:.2f}%\n")

print("--- MATRIZ DE CONFUSÃO ---")
print(f"Verdadeiros Negativos (Legítimos corretos):        {matriz[0][0]}")
print(f"Falsos Positivos      (Legítimos marcados como Phishing): {matriz[0][1]}")
print(f"Falsos Negativos      (Phishing marcado como Legítimo):   {matriz[1][0]}")
print(f"Verdadeiros Positivos (Phishing corretos):         {matriz[1][1]}\n")

print("--- RELATÓRIO DE CLASSIFICAÇÃO ---")
print(classification_report(y_test, y_pred, target_names=["Legítimo (0)", "Phishing (1)"]))
print("=" * 50)

      MÉTRICAS DE DESEMPENHO
Acurácia:  99.75%
Precisão:  98.12%
Recall:    91.29%
F1-Score:  94.58%

--- MATRIZ DE CONFUSÃO ---
Verdadeiros Negativos (Legítimos corretos):        199955
Falsos Positivos      (Legítimos marcados como Phishing): 87
Falsos Negativos      (Phishing marcado como Legítimo):   434
Verdadeiros Positivos (Phishing corretos):         4549

--- RELATÓRIO DE CLASSIFICAÇÃO ---
              precision    recall  f1-score   support

Legítimo (0)       1.00      1.00      1.00    200042
Phishing (1)       0.98      0.91      0.95      4983

    accuracy                           1.00    205025
   macro avg       0.99      0.96      0.97    205025
weighted avg       1.00      1.00      1.00    205025



In [8]:
# 8. Validação de URL Individual
DOMINIOS_CONFIAVEIS = [
    ".gov.br", ".edu.br", ".jus.br", ".mil.br", ".leg.br",
    "fgv.br", "youtube.com", "google.com", "instagram.com",
    "facebook.com", "whatsapp.com"
]

def eh_dominio_confiavel(url):
    url = url.lower()
    return any(dominio in url for dominio in DOMINIOS_CONFIAVEIS)

url_teste = "https://chatgpt.com/"

if eh_dominio_confiavel(url_teste):
    print("\nTeste de URL:", url_teste)
    print("Resultado: URL SEGURA (Whitelist)")
else:
    caracteristicas = pd.DataFrame([extract_features(url_teste)])
    previsao = modelo.predict(caracteristicas)[0]
    probabilidades = modelo.predict_proba(caracteristicas)[0]

    print("\nTeste de URL:", url_teste)
    if previsao == 0:
        print(f"Resultado: URL SEGURA (Confiança: {probabilidades[0] * 100:.1f}%)")
    else:
        print(f"Resultado: URL SUSPEITA (Confiança: {probabilidades[1] * 100:.1f}%)")


Teste de URL: https://chatgpt.com/
Resultado: URL SEGURA (Confiança: 100.0%)


In [20]:
# 9. Exportação do Arquivo .pkl
joblib.dump(modelo, "modelo.pkl")
print("\nArquivo 'modelo.pkl' gerado e pronto para o Android!")

files.download("modelo.pkl")


Arquivo 'modelo.pkl' gerado e pronto para o Android!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>